<a href="https://colab.research.google.com/github/hernandez-2007/INTELIGENCIA-ARTIFICAL-2/blob/main/tarea%20sesion%208.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis Predictivo del Costo de Viviendas en Bogotá

Este notebook tiene como objetivo desarrollar y comparar modelos predictivos para estimar el precio de casas o propiedades en Bogotá. Evaluaremos el rendimiento de la Regresión Lineal y los Árboles de Decisión para identificar cuál ofrece una mejor capacidad de predicción.

In [ ]:
# Importamos las librerías necesarias para el análisis de datos y la construcción de modelos.
import pandas as pd # pandas es fundamental para la manipulación y análisis de datos en tablas (DataFrames).
import numpy as np  # numpy se usa para operaciones numéricas y matemáticas eficientes.

from sklearn.model_selection import train_test_split # Para dividir el dataset en conjuntos de entrenamiento y prueba.
from sklearn.linear_model import LinearRegression # El modelo de Regresión Lineal.
from sklearn.tree import DecisionTreeRegressor    # El modelo de Árbol de Decisión para regresión.
from sklearn.metrics import mean_absolute_error, r2_score # Métricas para evaluar el rendimiento de los modelos.
from sklearn.preprocessing import OneHotEncoder # Para manejar variables categóricas.
from sklearn.compose import ColumnTransformer # Para aplicar transformaciones a columnas específicas.
from sklearn.impute import SimpleImputer # Para manejar valores faltantes.

import matplotlib.pyplot as plt # Para visualización de datos.
import seaborn as sns # Para visualización de datos más atractiva.

## 1. Carga y Exploración Inicial del Dataset

Primero, necesitamos cargar el conjunto de datos. **Asegúrate de que el archivo `bogota_house_prices.csv` esté disponible en la misma ubicación que este notebook o actualiza la ruta del archivo.**

In [ ]:
# Cargamos el dataset desde un archivo CSV. Asumo que el archivo se llama 'bogota_house_prices.csv'.
# Si tu archivo tiene otro nombre o está en otra ruta, por favor, ajusta 'ruta/a/tu/archivo.csv'.
try:
    df = pd.read_csv('bogota_house_prices.csv') # Intentamos leer el archivo CSV.
    print("Dataset cargado exitosamente.")
except FileNotFoundError:
    print("ERROR: El archivo 'bogota_house_prices.csv' no fue encontrado. "
          "Por favor, asegúrate de que el archivo exista en la misma carpeta "
          "o proporciona la ruta completa y correcta al archivo.")
    # Crear un DataFrame de ejemplo si el archivo no se encuentra para poder continuar con el código
    print("Creando un DataFrame de ejemplo para demostración.")
    data = {
        'area_mts2': [50, 75, 100, 120, 60, 90, 110, 80, 70, 130, 95, 85, 105, 65, 115],
        'habitaciones': [2, 3, 3, 4, 2, 3, 4, 3, 2, 4, 3, 3, 4, 2, 4],
        'banos': [1, 2, 2, 3, 1, 2, 3, 2, 1, 3, 2, 2, 3, 1, 3],
        'parqueaderos': [1, 1, 2, 2, 1, 1, 2, 1, 1, 2, 1, 1, 2, 1, 2],
        'localidad': ['Kennedy', 'Chapinero', 'Usaquén', 'Suba', 'Engativá', 'Chapinero', 'Usaquén', 'Kennedy', 'Suba', 'Usaquén', 'Suba', 'Chapinero', 'Kennedy', 'Engativá', 'Suba'],
        'estrato': [3, 4, 5, 4, 3, 4, 5, 3, 3, 5, 4, 4, 3, 3, 4],
        'precio_millones_cop': [250, 450, 600, 750, 300, 500, 680, 400, 350, 800, 550, 480, 620, 320, 700]
    }
    df = pd.DataFrame(data)


# Mostramos las primeras 5 filas del DataFrame para tener una idea de la estructura de los datos.
display(df.head())

In [ ]:
# Obtenemos un resumen de la información del DataFrame, incluyendo tipos de datos y valores no nulos.
# Esto es crucial para identificar columnas con datos faltantes (NaN) o tipos incorrectos.
display(df.info())

# Mostramos estadísticas descriptivas de las columnas numéricas.
# Esto nos da una idea de la distribución de los datos (media, desviación estándar, mínimos, máximos, cuartiles).
display(df.describe())

## 2. Preprocesamiento de Datos

En esta sección, prepararemos los datos para el modelado. Esto incluye el manejo de valores faltantes (si los hay) y la codificación de variables categóricas (como la 'localidad').

In [ ]:
# Verificamos la cantidad de valores nulos por columna.
# Es importante saber si hay datos faltantes y en qué columnas para decidir cómo manejarlos.
print("Valores nulos por columna:")
display(df.isnull().sum())

# Para este ejemplo, si hay valores nulos en columnas numéricas, los imputaremos con la media.
# Para columnas categóricas, podríamos imputar con la moda (el valor más frecuente).
# Aquí asumimos que nuestras columnas numéricas clave no tienen nulos o los manejaremos de forma simple.

# Separamos las características (X) del objetivo (y).
# 'precio_millones_cop' es la variable que queremos predecir, así que es nuestro objetivo (y).
# El resto de las columnas son las características (X) que usaremos para predecir el precio.
X = df.drop('precio_millones_cop', axis=1) # X contiene todas las columnas excepto 'precio_millones_cop'.
y = df['precio_millones_cop']           # y contiene solo la columna 'precio_millones_cop'.

In [ ]:
# Identificamos las columnas categóricas y numéricas para aplicar diferentes transformaciones.
categorical_features = X.select_dtypes(include=['object']).columns # Columnas con tipo de dato 'object' (generalmente strings).
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns # Columnas con tipos de dato numéricos.

# Creamos pipelines para preprocesar las columnas.
# Para las columnas numéricas, usaremos SimpleImputer para rellenar los valores faltantes con la media.
numerical_transformer = SimpleImputer(strategy='mean')

# Para las columnas categóricas, usaremos OneHotEncoder para convertirlas en un formato numérico.
# handle_unknown='ignore' asegura que si aparece una categoría nueva en el futuro, no cause error.
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Combinamos los transformadores usando ColumnTransformer.
# Esto nos permite aplicar diferentes transformaciones a diferentes columnas de X.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),     # Aplicar el imputador numérico a las columnas numéricas.
        ('cat', categorical_transformer, categorical_features) # Aplicar OneHotEncoder a las columnas categóricas.
    ])

# Aplicamos el preprocesador a nuestras características X.
# Esto transforma X, imputando nulos y codificando categóricas.
X_processed = preprocessor.fit_transform(X)

# Verificamos la forma de X_processed para asegurarnos de que la transformación funcionó.
# La cantidad de filas debe ser la misma, pero la cantidad de columnas puede haber cambiado debido a OneHotEncoder.
print(f"Forma de X después del preprocesamiento: {X_processed.shape}")

In [ ]:
# Dividimos los datos en conjuntos de entrenamiento y prueba.
# train_test_split toma X_processed (características) y y (objetivo).
# test_size=0.20 significa que el 20% de los datos se usarán para prueba y el 80% para entrenamiento.
# random_state=42 asegura que la división sea la misma cada vez que ejecutemos el código,
# lo que es útil para la reproducibilidad de los resultados.
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.20, random_state=42)

print(f"Tamaño del conjunto de entrenamiento de características (X_train): {X_train.shape}")
print(f"Tamaño del conjunto de prueba de características (X_test): {X_test.shape}")
print(f"Tamaño del conjunto de entrenamiento del objetivo (y_train): {y_train.shape}")
print(f"Tamaño del conjunto de prueba del objetivo (y_test): {y_test.shape}")